# N2: Bidirectional LSTM with Dual PhoBERT Layers

Test Bidirectional LSTM models with different PhoBERT embedding layers:
1. **Bidirectional LSTM Model 1**: TF-IDF + PhoBERT Layer 0 (before encoder)
2. **Bidirectional LSTM Model 1.1**: TF-IDF + PhoBERT Layer 0 + hand-crafted features
3. **Bidirectional LSTM Model 2**: TF-IDF + PhoBERT Layer 12 (after 12 encoders)
4. **Bidirectional LSTM Model 2.1**: TF-IDF + PhoBERT Layer 12 + hand-crafted features

Compare performance across all variants using Bidirectional LSTM architecture.

## 1. Import Required Libraries and Load Data

In [1]:
import numpy as np
import pandas as pd
import warnings
import torch
import torch.nn as nn
import torch.optim as optim
import gc
import time
from tqdm import tqdm
from torch.utils.data import TensorDataset, DataLoader
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, precision_score, recall_score
from transformers import AutoTokenizer, AutoModel

# Setup device for PyTorch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("✅ All imports successful")
print(f"   PyTorch Device: {device}")
print(f"   Using Bidirectional LSTM for neural networks")

✅ All imports successful
   PyTorch Device: cuda
   Using Bidirectional LSTM for neural networks


## 2. Load Data

In [2]:
train_path = '../../../data/splited/train_set.csv'
val_path = '../../../data/splited/val_set.csv'

df_train = pd.read_csv(train_path)
df_val = pd.read_csv(val_path)

y_train = df_train['label'].values
y_val = df_val['label'].values

print(f"✅ Data loaded: Train {df_train.shape} | Val {df_val.shape}")
print(f"   Labels: Train {(y_train==0).sum()} fake / {(y_train==1).sum()} real")
print(f"           Val {(y_val==0).sum()} fake / {(y_val==1).sum()} real")

✅ Data loaded: Train (3788, 28) | Val (474, 28)
   Labels: Train 3143 fake / 645 real
           Val 393 fake / 81 real


## 3. Generate TF-IDF Embeddings (900-dim)

In [3]:
texts_train_strict = df_train['text_strict'].fillna('').tolist()
texts_val_strict = df_val['text_strict'].fillna('').tolist()

tfidf = TfidfVectorizer(ngram_range=(1, 2), max_df=0.95, min_df=2, sublinear_tf=True)
X_train_tfidf_full = tfidf.fit_transform(texts_train_strict)
X_val_tfidf_full = tfidf.transform(texts_val_strict)

svd = TruncatedSVD(n_components=900, random_state=42)
X_train_tfidf = svd.fit_transform(X_train_tfidf_full)
X_val_tfidf = svd.transform(X_val_tfidf_full)

print(f"✅ TF-IDF embeddings: Train {X_train_tfidf.shape} | Val {X_val_tfidf.shape}")

✅ TF-IDF embeddings: Train (3788, 900) | Val (474, 900)


## 4. Extract PhoBERT Embeddings (Layer 0 & Layer 12)

In [4]:
texts_train_loose = df_train['text_loose'].fillna('').tolist()
texts_val_loose = df_val['text_loose'].fillna('').tolist()

# ✅ Load PhoBERT tokenizer and model ONCE (Pretrained model: vinai/phobert-base-v2)
print("📦 Loading PRETRAINED PhoBERT model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained('vinai/phobert-base-v2')
phobert_model = AutoModel.from_pretrained('vinai/phobert-base-v2',
                                           output_hidden_states=True).to(device).eval()
print("✅ PhoBERT (Pretrained) loaded successfully")

def extract_layer_embeddings(texts, tokenizer, model, layer_num=0, batch_size=16):
    """Extract PhoBERT embeddings from a specific layer (tokenizer + model passed in)"""
    embeddings = []
    
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc=f"Layer {layer_num}", leave=False):
            batch = texts[i:i+batch_size]
            inputs = tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=256)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = model(**inputs, output_hidden_states=True)
            
            # Get embeddings from specified layer
            hidden_states = outputs.hidden_states[layer_num]  # (batch, seq, hidden)
            
            # Use [CLS] token (first token)
            cls_tokens = hidden_states[:, 0, :].cpu().numpy()
            embeddings.extend(cls_tokens)
            del outputs, inputs
            torch.cuda.empty_cache()
    
    return np.array(embeddings)

# Extract BOTH Layer 0 and Layer 12 in one go
print("\n📊 Extracting PhoBERT Layer 0 (before encoder)...")
X_train_phobert_l0 = extract_layer_embeddings(texts_train_loose, tokenizer, phobert_model, layer_num=0, batch_size=16)
X_val_phobert_l0 = extract_layer_embeddings(texts_val_loose, tokenizer, phobert_model, layer_num=0, batch_size=16)
print(f"✅ PhoBERT Layer 0 embeddings: Train {X_train_phobert_l0.shape} | Val {X_val_phobert_l0.shape}")

print("\n📊 Extracting PhoBERT Layer 12 (after 12 encoders)...")
X_train_phobert_l12 = extract_layer_embeddings(texts_train_loose, tokenizer, phobert_model, layer_num=12, batch_size=16)
X_val_phobert_l12 = extract_layer_embeddings(texts_val_loose, tokenizer, phobert_model, layer_num=12, batch_size=16)
print(f"✅ PhoBERT Layer 12 embeddings: Train {X_train_phobert_l12.shape} | Val {X_val_phobert_l12.shape}")

# Clean up PhoBERT model to free memory
phobert_model.to('cpu')
del phobert_model, tokenizer
gc.collect()
print("✅ PhoBERT model released from GPU")

📦 Loading PRETRAINED PhoBERT model and tokenizer...


Some weights of RobertaModel were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ PhoBERT (Pretrained) loaded successfully

📊 Extracting PhoBERT Layer 0 (before encoder)...


✅ PhoBERT Layer 0 embeddings: Train (3788, 768) | Val (474, 768)

📊 Extracting PhoBERT Layer 12 (after 12 encoders)...


✅ PhoBERT Layer 12 embeddings: Train (3788, 768) | Val (474, 768)
✅ PhoBERT model released from GPU


## 5. Extract Hand-Crafted Features & Define Bidirectional LSTM Architecture

In [5]:
exclude_cols = {'id', 'label', 'text_strict', 'text_loose', 'post_message'}
numeric_cols = df_train.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [col for col in numeric_cols 
                if col not in exclude_cols and col.strip() != '' and not col.startswith('Unnamed')]

if feature_cols:
    X_train_features = df_train[feature_cols].fillna(0).values
    X_val_features = df_val[feature_cols].fillna(0).values
else:
    X_train_features = np.array([]).reshape(len(df_train), 0)
    X_val_features = np.array([]).reshape(len(df_val), 0)

# Normalize hand-crafted features
scaler_features = StandardScaler()
X_train_features_scaled = scaler_features.fit_transform(X_train_features) if X_train_features.shape[1] > 0 else X_train_features
X_val_features_scaled = scaler_features.transform(X_val_features) if X_val_features.shape[1] > 0 else X_val_features

print(f"✅ Hand-crafted features: {len(feature_cols)} features")
print(f"   Train {X_train_features_scaled.shape} | Val {X_val_features_scaled.shape}")

# Define Bidirectional LSTM architecture
class BidirectionalLSTMNet(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, dropout=0.5):
        super(BidirectionalLSTMNet, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 2)
        )
    
    def forward(self, x):
        # x shape: (batch, seq_len=1, input_dim)
        lstm_out, (h_n, c_n) = self.lstm(x)  # h_n: (num_layers*2, batch, hidden_dim) for bidirectional
        # Use last hidden state from top layer (average forward and backward)
        last_hidden_forward = h_n[-2, :, :]  # Forward direction from top layer
        last_hidden_backward = h_n[-1, :, :]  # Backward direction from top layer
        last_hidden = torch.cat([last_hidden_forward, last_hidden_backward], dim=1)  # Concatenate: (batch, hidden_dim*2)
        out = self.fc(last_hidden)  # (batch, 2)
        return out

print("✅ Bidirectional LSTM architecture defined")

✅ Hand-crafted features: 23 features
   Train (3788, 23) | Val (474, 23)
✅ Bidirectional LSTM architecture defined


## 6. Model 1: Bidirectional LSTM with TF-IDF + PhoBERT Layer 0

In [6]:
# Combine: TF-IDF + PhoBERT Layer 0
X_train_model1 = np.hstack([X_train_tfidf, X_train_phobert_l0])
X_val_model1 = np.hstack([X_val_tfidf, X_val_phobert_l0])

print(f"Model 1 input (TF + Layer 0): Train={X_train_model1.shape}, Val={X_val_model1.shape}")

# Create tensors and DataLoaders - reshape for LSTM (batch, seq_len=1, features)
train_dataset_m1 = TensorDataset(torch.from_numpy(X_train_model1).float().unsqueeze(1),
                                 torch.from_numpy(y_train).long())
val_dataset_m1 = TensorDataset(torch.from_numpy(X_val_model1).float().unsqueeze(1),
                               torch.from_numpy(y_val).long())

train_loader_m1 = DataLoader(train_dataset_m1, batch_size=32, shuffle=True)
val_loader_m1 = DataLoader(val_dataset_m1, batch_size=32, shuffle=False)

# Initialize and train Bidirectional LSTM Model 1
model1 = BidirectionalLSTMNet(input_dim=X_train_model1.shape[1]).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model1.parameters(), lr=0.001)

print("🔄 Training Model 1 (TF + PhoBERT Layer 0)...")
start_time = time.time()

for epoch in range(15):
    # Training
    model1.train()
    train_loss = 0
    for X_batch, y_batch in train_loader_m1:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        pred = model1(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    # Validation
    model1.eval()
    val_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in val_loader_m1:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            pred = model1(X_batch)
            loss = criterion(pred, y_batch)
            val_loss += loss.item()
            all_preds.extend(pred.argmax(1).cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    
    val_acc = accuracy_score(all_labels, all_preds)
    if (epoch + 1) % 5 == 0:
        print(f"  Epoch {epoch+1:2d}: Train Loss={train_loss/len(train_loader_m1):.4f}, "
              f"Val Loss={val_loss/len(val_loader_m1):.4f}, Val Acc={val_acc:.4f}")

train_time_m1 = time.time() - start_time
print(f"✅ Model 1 training completed in {train_time_m1:.2f}s\n")

Model 1 input (TF + Layer 0): Train=(3788, 1668), Val=(474, 1668)
🔄 Training Model 1 (TF + PhoBERT Layer 0)...
  Epoch  5: Train Loss=0.2504, Val Loss=0.2443, Val Acc=0.9030
  Epoch 10: Train Loss=0.1917, Val Loss=0.2288, Val Acc=0.9093
  Epoch 15: Train Loss=0.1608, Val Loss=0.2815, Val Acc=0.9093
✅ Model 1 training completed in 8.32s



### 6.1 Evaluate Model 1

In [7]:
print("\n" + "="*80)
print("MODEL 1: EVALUATION (TF-IDF + PhoBERT Layer 0)")
print("="*80)

# Evaluation
model1.eval()
with torch.no_grad():
    X_val_m1_tensor = torch.from_numpy(X_val_model1).float().unsqueeze(1).to(device)
    outputs_val = model1(X_val_m1_tensor)
    y_val_pred_m1 = outputs_val.argmax(dim=1).cpu().numpy()
    y_val_proba_m1 = torch.softmax(outputs_val, dim=1)[:, 1].cpu().numpy()

# Calculate metrics
f1_m1 = f1_score(y_val, y_val_pred_m1, average='weighted')
auc_m1 = roc_auc_score(y_val, y_val_proba_m1)
acc_m1 = accuracy_score(y_val, y_val_pred_m1)
prec_m1 = precision_score(y_val, y_val_pred_m1, average='weighted')
rec_m1 = recall_score(y_val, y_val_pred_m1, average='weighted')

print(f"\n✅ Model 1 Metrics:")
print(f"   F1 Score:  {f1_m1:.4f}")
print(f"   AUC Score: {auc_m1:.4f}")
print(f"   Accuracy:  {acc_m1:.4f}")
print(f"   Precision: {prec_m1:.4f}")
print(f"   Recall:    {rec_m1:.4f}")

results_model1 = {
    'F1': f1_m1,
    'AUC': auc_m1,
    'Accuracy': acc_m1,
    'Precision': prec_m1,
    'Recall': rec_m1
}


MODEL 1: EVALUATION (TF-IDF + PhoBERT Layer 0)

✅ Model 1 Metrics:
   F1 Score:  0.9017
   AUC Score: 0.9291
   Accuracy:  0.9093
   Precision: 0.9056
   Recall:    0.9093


## 7. Model 1.1: Bidirectional LSTM with TF-IDF + Layer 0 + Hand-Crafted Features

In [8]:
# Combine: TF-IDF + PhoBERT Layer 0 + Hand-crafted
X_train_model1_1 = np.hstack([X_train_tfidf, X_train_phobert_l0, X_train_features_scaled])
X_val_model1_1 = np.hstack([X_val_tfidf, X_val_phobert_l0, X_val_features_scaled])

print(f"Model 1.1 input (TF + Layer 0 + HC): Train={X_train_model1_1.shape}, Val={X_val_model1_1.shape}")

# Create tensors and DataLoaders
train_dataset_m1_1 = TensorDataset(torch.from_numpy(X_train_model1_1).float().unsqueeze(1),
                                   torch.from_numpy(y_train).long())
val_dataset_m1_1 = TensorDataset(torch.from_numpy(X_val_model1_1).float().unsqueeze(1),
                                 torch.from_numpy(y_val).long())

train_loader_m1_1 = DataLoader(train_dataset_m1_1, batch_size=32, shuffle=True)
val_loader_m1_1 = DataLoader(val_dataset_m1_1, batch_size=32, shuffle=False)

# Initialize and train Bidirectional LSTM Model 1.1
model1_1 = BidirectionalLSTMNet(input_dim=X_train_model1_1.shape[1]).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model1_1.parameters(), lr=0.001)

print("🔄 Training Model 1.1 (TF + PhoBERT Layer 0 + Hand-Crafted)...")
start_time = time.time()

for epoch in range(15):
    # Training
    model1_1.train()
    train_loss = 0
    for X_batch, y_batch in train_loader_m1_1:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        pred = model1_1(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    # Validation
    model1_1.eval()
    val_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in val_loader_m1_1:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            pred = model1_1(X_batch)
            loss = criterion(pred, y_batch)
            val_loss += loss.item()
            all_preds.extend(pred.argmax(1).cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    
    val_acc = accuracy_score(all_labels, all_preds)
    if (epoch + 1) % 5 == 0:
        print(f"  Epoch {epoch+1:2d}: Train Loss={train_loss/len(train_loader_m1_1):.4f}, "
              f"Val Loss={val_loss/len(val_loader_m1_1):.4f}, Val Acc={val_acc:.4f}")

train_time_m1_1 = time.time() - start_time
print(f"✅ Model 1.1 training completed in {train_time_m1_1:.2f}s\n")

Model 1.1 input (TF + Layer 0 + HC): Train=(3788, 1691), Val=(474, 1691)
🔄 Training Model 1.1 (TF + PhoBERT Layer 0 + Hand-Crafted)...
  Epoch  5: Train Loss=0.2483, Val Loss=0.2652, Val Acc=0.9030
  Epoch 10: Train Loss=0.1605, Val Loss=0.2644, Val Acc=0.9051
  Epoch 15: Train Loss=0.1194, Val Loss=0.2852, Val Acc=0.9114
✅ Model 1.1 training completed in 8.35s



### 7.1 Evaluate Model 1.1

In [9]:
print("\n" + "="*80)
print("MODEL 1.1: EVALUATION (TF-IDF + PhoBERT Layer 0 + Hand-Crafted)")
print("="*80)

# Evaluation
model1_1.eval()
with torch.no_grad():
    X_val_m1_1_tensor = torch.from_numpy(X_val_model1_1).float().unsqueeze(1).to(device)
    outputs_val = model1_1(X_val_m1_1_tensor)
    y_val_pred_m1_1 = outputs_val.argmax(dim=1).cpu().numpy()
    y_val_proba_m1_1 = torch.softmax(outputs_val, dim=1)[:, 1].cpu().numpy()

# Calculate metrics
f1_m1_1 = f1_score(y_val, y_val_pred_m1_1, average='weighted')
auc_m1_1 = roc_auc_score(y_val, y_val_proba_m1_1)
acc_m1_1 = accuracy_score(y_val, y_val_pred_m1_1)
prec_m1_1 = precision_score(y_val, y_val_pred_m1_1, average='weighted')
rec_m1_1 = recall_score(y_val, y_val_pred_m1_1, average='weighted')

print(f"\n✅ Model 1.1 Metrics:")
print(f"   F1 Score:  {f1_m1_1:.4f}")
print(f"   AUC Score: {auc_m1_1:.4f}")
print(f"   Accuracy:  {acc_m1_1:.4f}")
print(f"   Precision: {prec_m1_1:.4f}")
print(f"   Recall:    {rec_m1_1:.4f}")

results_model1_1 = {
    'F1': f1_m1_1,
    'AUC': auc_m1_1,
    'Accuracy': acc_m1_1,
    'Precision': prec_m1_1,
    'Recall': rec_m1_1
}


MODEL 1.1: EVALUATION (TF-IDF + PhoBERT Layer 0 + Hand-Crafted)

✅ Model 1.1 Metrics:
   F1 Score:  0.9074
   AUC Score: 0.9319
   Accuracy:  0.9114
   Precision: 0.9070
   Recall:    0.9114


## 8. Model 2: Bidirectional LSTM with TF-IDF + PhoBERT Layer 12

In [10]:
# Combine: TF-IDF + PhoBERT Layer 12
X_train_model2 = np.hstack([X_train_tfidf, X_train_phobert_l12])
X_val_model2 = np.hstack([X_val_tfidf, X_val_phobert_l12])

print(f"Model 2 input (TF + Layer 12): Train={X_train_model2.shape}, Val={X_val_model2.shape}")

# Create tensors and DataLoaders
train_dataset_m2 = TensorDataset(torch.from_numpy(X_train_model2).float().unsqueeze(1),
                                 torch.from_numpy(y_train).long())
val_dataset_m2 = TensorDataset(torch.from_numpy(X_val_model2).float().unsqueeze(1),
                               torch.from_numpy(y_val).long())

train_loader_m2 = DataLoader(train_dataset_m2, batch_size=32, shuffle=True)
val_loader_m2 = DataLoader(val_dataset_m2, batch_size=32, shuffle=False)

# Initialize and train Bidirectional LSTM Model 2
model2 = BidirectionalLSTMNet(input_dim=X_train_model2.shape[1]).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model2.parameters(), lr=0.001)

print("🔄 Training Model 2 (TF + PhoBERT Layer 12)...")
start_time = time.time()

for epoch in range(15):
    # Training
    model2.train()
    train_loss = 0
    for X_batch, y_batch in train_loader_m2:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        pred = model2(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    # Validation
    model2.eval()
    val_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in val_loader_m2:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            pred = model2(X_batch)
            loss = criterion(pred, y_batch)
            val_loss += loss.item()
            all_preds.extend(pred.argmax(1).cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    
    val_acc = accuracy_score(all_labels, all_preds)
    if (epoch + 1) % 5 == 0:
        print(f"  Epoch {epoch+1:2d}: Train Loss={train_loss/len(train_loader_m2):.4f}, "
              f"Val Loss={val_loss/len(val_loader_m2):.4f}, Val Acc={val_acc:.4f}")

train_time_m2 = time.time() - start_time
print(f"✅ Model 2 training completed in {train_time_m2:.2f}s\n")

Model 2 input (TF + Layer 12): Train=(3788, 1668), Val=(474, 1668)
🔄 Training Model 2 (TF + PhoBERT Layer 12)...
  Epoch  5: Train Loss=0.1908, Val Loss=0.2174, Val Acc=0.9114
  Epoch 10: Train Loss=0.1055, Val Loss=0.2735, Val Acc=0.9008
  Epoch 15: Train Loss=0.0569, Val Loss=0.3541, Val Acc=0.8987
✅ Model 2 training completed in 7.91s



### 8.1 Evaluate Model 2

In [11]:
print("\n" + "="*80)
print("MODEL 2: EVALUATION (TF-IDF + PhoBERT Layer 12)")
print("="*80)

# Evaluation
model2.eval()
with torch.no_grad():
    X_val_m2_tensor = torch.from_numpy(X_val_model2).float().unsqueeze(1).to(device)
    outputs_val = model2(X_val_m2_tensor)
    y_val_pred_m2 = outputs_val.argmax(dim=1).cpu().numpy()
    y_val_proba_m2 = torch.softmax(outputs_val, dim=1)[:, 1].cpu().numpy()

# Calculate metrics
f1_m2 = f1_score(y_val, y_val_pred_m2, average='weighted')
auc_m2 = roc_auc_score(y_val, y_val_proba_m2)
acc_m2 = accuracy_score(y_val, y_val_pred_m2)
prec_m2 = precision_score(y_val, y_val_pred_m2, average='weighted')
rec_m2 = recall_score(y_val, y_val_pred_m2, average='weighted')

print(f"\n✅ Model 2 Metrics:")
print(f"   F1 Score:  {f1_m2:.4f}")
print(f"   AUC Score: {auc_m2:.4f}")
print(f"   Accuracy:  {acc_m2:.4f}")
print(f"   Precision: {prec_m2:.4f}")
print(f"   Recall:    {rec_m2:.4f}")

results_model2 = {
    'F1': f1_m2,
    'AUC': auc_m2,
    'Accuracy': acc_m2,
    'Precision': prec_m2,
    'Recall': rec_m2
}


MODEL 2: EVALUATION (TF-IDF + PhoBERT Layer 12)

✅ Model 2 Metrics:
   F1 Score:  0.8935
   AUC Score: 0.9533
   Accuracy:  0.8987
   Precision: 0.8927
   Recall:    0.8987


## 9. Model 2.1: Bidirectional LSTM with TF-IDF + Layer 12 + Hand-Crafted Features

In [16]:
# Combine: TF-IDF + PhoBERT Layer 12 + Hand-crafted
X_train_model2_1 = np.hstack([X_train_tfidf, X_train_phobert_l12, X_train_features_scaled])
X_val_model2_1 = np.hstack([X_val_tfidf, X_val_phobert_l12, X_val_features_scaled])

print(f"Model 2.1 input (TF + Layer 12 + HC): Train={X_train_model2_1.shape}, Val={X_val_model2_1.shape}")

# Create tensors and DataLoaders
train_dataset_m2_1 = TensorDataset(torch.from_numpy(X_train_model2_1).float().unsqueeze(1),
                                   torch.from_numpy(y_train).long())
val_dataset_m2_1 = TensorDataset(torch.from_numpy(X_val_model2_1).float().unsqueeze(1),
                                 torch.from_numpy(y_val).long())

train_loader_m2_1 = DataLoader(train_dataset_m2_1, batch_size=32, shuffle=True)
val_loader_m2_1 = DataLoader(val_dataset_m2_1, batch_size=32, shuffle=False)

# Initialize and train Bidirectional LSTM Model 2.1
model2_1 = BidirectionalLSTMNet(input_dim=X_train_model2_1.shape[1]).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model2_1.parameters(), lr=0.001)

print("🔄 Training Model 2.1 (TF + PhoBERT Layer 12 + Hand-Crafted)...")
start_time = time.time()

for epoch in range(15):
    # Training
    model2_1.train()
    train_loss = 0
    for X_batch, y_batch in train_loader_m2_1:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        pred = model2_1(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    # Validation
    model2_1.eval()
    val_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in val_loader_m2_1:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            pred = model2_1(X_batch)
            loss = criterion(pred, y_batch)
            val_loss += loss.item()
            all_preds.extend(pred.argmax(1).cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    
    val_acc = accuracy_score(all_labels, all_preds)
    if (epoch + 1) % 5 == 0:
        print(f"  Epoch {epoch+1:2d}: Train Loss={train_loss/len(train_loader_m2_1):.4f}, "
              f"Val Loss={val_loss/len(val_loader_m2_1):.4f}, Val Acc={val_acc:.4f}")

train_time_m2_1 = time.time() - start_time
print(f"✅ Model 2.1 training completed in {train_time_m2_1:.2f}s\n")

Model 2.1 input (TF + Layer 12 + HC): Train=(3788, 1691), Val=(474, 1691)
🔄 Training Model 2.1 (TF + PhoBERT Layer 12 + Hand-Crafted)...
  Epoch  5: Train Loss=0.1584, Val Loss=0.1935, Val Acc=0.9051
  Epoch 10: Train Loss=0.0824, Val Loss=0.3335, Val Acc=0.9093
  Epoch 15: Train Loss=0.0427, Val Loss=0.3371, Val Acc=0.9030
✅ Model 2.1 training completed in 7.94s



### 9.1 Evaluate Model 2.1

In [17]:
print("\n" + "="*80)
print("MODEL 2.1: EVALUATION (TF-IDF + PhoBERT Layer 12 + Hand-Crafted)")
print("="*80)

# Evaluation
model2_1.eval()
with torch.no_grad():
    X_val_m2_1_tensor = torch.from_numpy(X_val_model2_1).float().unsqueeze(1).to(device)
    outputs_val = model2_1(X_val_m2_1_tensor)
    y_val_pred_m2_1 = outputs_val.argmax(dim=1).cpu().numpy()
    y_val_proba_m2_1 = torch.softmax(outputs_val, dim=1)[:, 1].cpu().numpy()

# Calculate metrics
f1_m2_1 = f1_score(y_val, y_val_pred_m2_1, average='weighted')
auc_m2_1 = roc_auc_score(y_val, y_val_proba_m2_1)
acc_m2_1 = accuracy_score(y_val, y_val_pred_m2_1)
prec_m2_1 = precision_score(y_val, y_val_pred_m2_1, average='weighted')
rec_m2_1 = recall_score(y_val, y_val_pred_m2_1, average='weighted')

print(f"\n✅ Model 2.1 Metrics:")
print(f"   F1 Score:  {f1_m2_1:.4f}")
print(f"   AUC Score: {auc_m2_1:.4f}")
print(f"   Accuracy:  {acc_m2_1:.4f}")
print(f"   Precision: {prec_m2_1:.4f}")
print(f"   Recall:    {rec_m2_1:.4f}")

results_model2_1 = {
    'F1': f1_m2_1,
    'AUC': auc_m2_1,
    'Accuracy': acc_m2_1,
    'Precision': prec_m2_1,
    'Recall': rec_m2_1
}


MODEL 2.1: EVALUATION (TF-IDF + PhoBERT Layer 12 + Hand-Crafted)

✅ Model 2.1 Metrics:
   F1 Score:  0.9039
   AUC Score: 0.9542
   Accuracy:  0.9030
   Precision: 0.9050
   Recall:    0.9030


## 10. Comparison: All 4 Bidirectional LSTM Models vs Unidirectional LSTM

In [18]:
print("\n" + "="*80)
print("COMPREHENSIVE COMPARISON: ALL 4 BIDIRECTIONAL LSTM MODELS")
print("="*80)

# Create comprehensive comparison DataFrame with all metrics
comparison_data = {
    'Model': [
        'BiModel 1 (Layer 0)',
        'BiModel 1.1 (Layer 0 + HC)',
        'BiModel 2 (Layer 12)',
        'BiModel 2.1 (Layer 12 + HC)'
    ],
    'PhoBERT Layer': [0, 0, 12, 12],
    'Hand-Crafted': ['No', 'Yes', 'No', 'Yes'],
    'F1': [
        f"{results_model1['F1']:.4f}",
        f"{results_model1_1['F1']:.4f}",
        f"{results_model2['F1']:.4f}",
        f"{results_model2_1['F1']:.4f}"
    ],
    'AUC': [
        f"{results_model1['AUC']:.4f}",
        f"{results_model1_1['AUC']:.4f}",
        f"{results_model2['AUC']:.4f}",
        f"{results_model2_1['AUC']:.4f}"
    ],
    'Accuracy': [
        f"{results_model1['Accuracy']:.4f}",
        f"{results_model1_1['Accuracy']:.4f}",
        f"{results_model2['Accuracy']:.4f}",
        f"{results_model2_1['Accuracy']:.4f}"
    ],
    'Precision': [
        f"{results_model1['Precision']:.4f}",
        f"{results_model1_1['Precision']:.4f}",
        f"{results_model2['Precision']:.4f}",
        f"{results_model2_1['Precision']:.4f}"
    ],
    'Recall': [
        f"{results_model1['Recall']:.4f}",
        f"{results_model1_1['Recall']:.4f}",
        f"{results_model2['Recall']:.4f}",
        f"{results_model2_1['Recall']:.4f}"
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print("\n" + "─"*140)
print("PERFORMANCE METRICS SUMMARY - ALL 4 BIDIRECTIONAL LSTM MODELS")
print("─"*140 + "\n")
print(comparison_df.to_string(index=False))

# Collect numeric scores for analysis
f1_scores = [results_model1['F1'], results_model1_1['F1'], results_model2['F1'], results_model2_1['F1']]
auc_scores = [results_model1['AUC'], results_model1_1['AUC'], results_model2['AUC'], results_model2_1['AUC']]
accuracy_scores = [results_model1['Accuracy'], results_model1_1['Accuracy'], results_model2['Accuracy'], results_model2_1['Accuracy']]
precision_scores = [results_model1['Precision'], results_model1_1['Precision'], results_model2['Precision'], results_model2_1['Precision']]
recall_scores = [results_model1['Recall'], results_model1_1['Recall'], results_model2['Recall'], results_model2_1['Recall']]

best_f1_idx = np.argmax(f1_scores)
best_auc_idx = np.argmax(auc_scores)
best_accuracy_idx = np.argmax(accuracy_scores)

print(f"\n" + "="*80)
print("BEST PERFORMERS")
print("="*80)
print(f"\n🏆 Best F1 Score: {comparison_data['Model'][best_f1_idx]}")
print(f"   F1={f1_scores[best_f1_idx]:.4f} | AUC={auc_scores[best_f1_idx]:.4f} | Prec={precision_scores[best_f1_idx]:.4f} | Rec={recall_scores[best_f1_idx]:.4f}")

print(f"\n🏆 Best AUC Score: {comparison_data['Model'][best_auc_idx]}")
print(f"   F1={f1_scores[best_auc_idx]:.4f} | AUC={auc_scores[best_auc_idx]:.4f} | Prec={precision_scores[best_auc_idx]:.4f} | Rec={recall_scores[best_auc_idx]:.4f}")

print(f"\n🏆 Best Accuracy: {comparison_data['Model'][best_accuracy_idx]}")
print(f"   Acc={accuracy_scores[best_accuracy_idx]:.4f} | F1={f1_scores[best_accuracy_idx]:.4f}")

print(f"\n✅ Comprehensive Comparison Complete!")


COMPREHENSIVE COMPARISON: ALL 4 BIDIRECTIONAL LSTM MODELS

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
PERFORMANCE METRICS SUMMARY - ALL 4 BIDIRECTIONAL LSTM MODELS
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

                      Model  PhoBERT Layer Hand-Crafted     F1    AUC Accuracy Precision Recall
        BiModel 1 (Layer 0)              0           No 0.9017 0.9291   0.9093    0.9056 0.9093
 BiModel 1.1 (Layer 0 + HC)              0          Yes 0.9074 0.9319   0.9114    0.9070 0.9114
       BiModel 2 (Layer 12)             12           No 0.8935 0.9533   0.8987    0.8927 0.8987
BiModel 2.1 (Layer 12 + HC)             12          Yes 0.9039 0.9542   0.9030    0.9050 0.9030

BEST PERFORMERS

🏆 Best F1 Score: BiModel 1.1 (Layer 0 + HC)
   F1=0.9074 | AUC=0.9319 | Prec=0.9070 | Rec=0.9114


## 11. Model Complexity & Computational Analysis

In [19]:
def count_bidirectional_lstm_params(input_dim, hidden_dim=128, num_layers=2):
    """Count parameters in Bidirectional LSTM with detailed breakdown"""
    lstm_params = 0
    prev_dim = input_dim
    for layer in range(num_layers):
        # Bidirectional has 2x: forward + backward
        lstm_params += 2 * 4 * ((prev_dim + hidden_dim) * hidden_dim + hidden_dim)
        prev_dim = hidden_dim
    
    # FC layers (input is hidden_dim*2 for bidirectional)
    fc_params = (hidden_dim * 2 * 64) + 64 + (64 * 2) + 2
    return lstm_params + fc_params

def count_bidirectional_lstm_flops(input_dim, hidden_dim=128, num_layers=2, seq_len=1):
    """Estimate FLOPs for Bidirectional LSTM forward pass per sample"""
    lstm_flops = 0
    prev_dim = input_dim
    for layer in range(num_layers):
        # Bidirectional: 2x FLOPs (forward + backward)
        lstm_flops += seq_len * 2 * 4 * 2 * ((prev_dim + hidden_dim) * hidden_dim)
        prev_dim = hidden_dim
    
    # FC FLOPs
    fc_flops = 2 * (hidden_dim * 2 * 64) + 2 * (64 * 2)
    return lstm_flops + fc_flops

print("\n" + "="*80)
print("BIDIRECTIONAL LSTM MODEL COMPLEXITY & COMPUTATIONAL ANALYSIS")
print("="*80)

# Calculate parameters and FLOPs for all models
input_dims = [1668, 1668 + X_train_features_scaled.shape[1], 1668, 1668 + X_train_features_scaled.shape[1]]
lstm_params = []
lstm_flops = []

for inp_dim in input_dims:
    params = count_bidirectional_lstm_params(inp_dim)
    flops = count_bidirectional_lstm_flops(inp_dim)
    lstm_params.append(params)
    lstm_flops.append(flops)

complexity_data_lstm = {
    'Model': [
        'BiModel 1 (Layer 0)',
        'BiModel 1.1 (Layer 0 + HC)',
        'BiModel 2 (Layer 12)',
        'BiModel 2.1 (Layer 12 + HC)'
    ],
    'Input Dims': input_dims,
    'Parameters': lstm_params,
    'FLOPs/Sample': lstm_flops,
    'F1 Score': f1_scores,
    'AUC Score': auc_scores,
    'Accuracy': accuracy_scores,
    'Precision': precision_scores,
    'Recall': recall_scores
}

print("\n" + "─"*140)
print("TABLE 1: BIDIRECTIONAL LSTM MODELS - COMPLEXITY METRICS")
print("─"*140 + "\n")

complexity_display_df = pd.DataFrame({
    'Model': complexity_data_lstm['Model'],
    'Input Dims': complexity_data_lstm['Input Dims'],
    'Parameters': [f"{int(p):,}" for p in lstm_params],
    'FLOPs/Sample': [f"{int(f):,}" for f in lstm_flops],
    'F1 Score': [f"{f:.4f}" for f in f1_scores],
    'AUC Score': [f"{a:.4f}" for a in auc_scores]
})
print(complexity_display_df.to_string(index=False))

print("\n" + "─"*140)
print("TABLE 2: COMPLETE PERFORMANCE METRICS")
print("─"*140 + "\n")

detailed_df = pd.DataFrame({
    'Model': complexity_data_lstm['Model'],
    'F1': [f"{f:.4f}" for f in f1_scores],
    'AUC': [f"{a:.4f}" for a in auc_scores],
    'Accuracy': [f"{a:.4f}" for a in accuracy_scores],
    'Precision': [f"{p:.4f}" for p in precision_scores],
    'Recall': [f"{r:.4f}" for r in recall_scores]
})
print(detailed_df.to_string(index=False))

print("\n" + "="*80)
print("NOTES ON BIDIRECTIONAL LSTM ARCHITECTURE")
print("-" * 80)
print("• Architecture: 2-layer Bidirectional LSTM (128 hidden units) + FC[256→64→2]")
print("• Key Difference: Bidirectional LSTM reads sequence in both directions")
print("• Forward LSTM: processes sequence left-to-right")
print("• Backward LSTM: processes sequence right-to-left")
print("• Output: Concatenation of forward and backward hidden states (hidden_dim*2)")
print("• Parameters: 2x unidirectional LSTM (forward + backward gates)")
print("• FLOPs: 2x unidirectional LSTM")
print("• Advantage: Captures context from both directions → better for NLP")
print("="*80)

print(f"\n✅ Complexity Analysis Complete!")


BIDIRECTIONAL LSTM MODEL COMPLEXITY & COMPUTATIONAL ANALYSIS

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
TABLE 1: BIDIRECTIONAL LSTM MODELS - COMPLEXITY METRICS
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

                      Model  Input Dims Parameters FLOPs/Sample F1 Score AUC Score
        BiModel 1 (Layer 0)        1668  2,119,874    4,235,520   0.9017    0.9291
 BiModel 1.1 (Layer 0 + HC)        1691  2,143,426    4,282,624   0.9074    0.9319
       BiModel 2 (Layer 12)        1668  2,119,874    4,235,520   0.8935    0.9533
BiModel 2.1 (Layer 12 + HC)        1691  2,143,426    4,282,624   0.9039    0.9542

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
TABLE 2: COMPLETE PERFORMANCE METRICS
───